In [3]:
from scraper.scraper import run_scraper
from scraper.data_transform import clean_data, get_available_dates, remove_brackets
import pandas as pd
from db.db_write import setup_database_connection, write_to_db
from secret import USER, PASSWORD, HOST, PORT

URL_DICT = {
    'Cafeteria Wilhelmstraße': 'https://www.my-stuwe.de//wp-json/mealplans/v1/canteens/715?lang=de&v=1731244959433',
    'Cafeteria Morgenstelle': 'https://www.my-stuwe.de//wp-json/mealplans/v1/canteens/724?lang=de&v=1731245000291',
    'Cafeteria und Mensa Prinz Karl': 'https://www.my-stuwe.de//wp-json/mealplans/v1/canteens/623?lang=de&v=1731088441410',
    'Mensa Wilhelmstraße': 'https://www.my-stuwe.de//wp-json/mealplans/v1/canteens/611?lang=de&v=1731088386173',
    'Mensa Morgenstelle': 'https://www.my-stuwe.de//wp-json/mealplans/v1/canteens/621?lang=de&v=1731088361352'
}

def main():
    engine, Session = setup_database_connection(USER, PASSWORD, HOST, PORT)
    result_df = pd.DataFrame(columns=["menuDate", "menuLine", "menu", "studentPrice", "location"])

    for option in URL_DICT:
        temp_df = run_scraper(option, URL_DICT)
        cleaned_df = clean_data(temp_df, option)
        result_df = pd.concat([result_df, cleaned_df])

    available_dates = get_available_dates()
    filtered_df = result_df[result_df["menuDate"].isin(available_dates)].copy()

    filtered_df["menu"] = filtered_df['menu'].apply(remove_brackets)

    write_to_db(filtered_df, engine, Session)

    print("Finished")


df = main()


Successfully connected to database
Finished


In [2]:
df

,menuDate,menuLine,menu,studentPrice,location,id,guestPrice,pupilPrice,meats,icons,allergens,additives
3,2024-12-17,Angebot des Tages,"Pasta hausgemacht-Conchiglie mit, veganem Aube...","5,60",Cafeteria Wilhelmstraße,233,"5,60","5,60",NA,NA,"So, ML, Gl-a","1, 7"
4,2024-12-18,Angebot des Tages,"Pizza Tonno mit Thunfisch und Zweibeln oder, P...","6,50",Cafeteria Wilhelmstraße,234,"6,50","6,50",NA,NA,"ML, Gl-a, Gl-b, Nu-c",11
5,2024-12-19,Angebot des Tages,"Pasta hausgemacht-Papardelle mit, Wildsugo od...","6,50",Cafeteria Wilhelmstraße,235,"6,50","6,50",W,W,"ML, Sf, Gl-a, ALK",4
2,2024-12-16,Angebot des Tages,Pommes frites,"1,70",Cafeteria Morgenstelle,337,"1,70","1,70",NA,NA,NA,9
3,2024-12-16,Angebot des Tages,Pizza Salami,"6,50",Cafeteria Morgenstelle,338,"6,50","6,50",NA,NA,"Sn, ML, Gl-a, Gl-b","2, 3, 4, 11"
...,...,...,...,...,...,...,...,...,...,...,...,...
31,2024-12-19,Dessert vorport.,Dessertauswahltheke,"1,05",Mensa Morgenstelle,310,"1,65","1,65",NA,NA,ML,NA
34,2024-12-16,Dessert SB,Frisches Obst,"0,75",Mensa Morgenstelle,324,"0,75","0,75",NA,NA,NA,NA
35,2024-12-17,Dessert SB,Frisches Obst,"0,75",Mensa Morgenstelle,325,"0,75","0,75",NA,NA,NA,NA
36,2024-12-18,Dessert SB,Frisches Obst,"0,75",Mensa Morgenstelle,326,"0,75","0,75",NA,NA,NA,NA
